# Tiny Siamese + SECOND — 12 конфигураций, 15 → 25 → 40 → 65 эпох

**Запуск:** сохрани notebook в `notebooks/`, выбери kernel проекта, затем
**Restart Kernel → Run All**. `MODE="train_all"` уже включён. Старое обучение сначала
мягко останови и дождись завершения его workers: новый поиск использует те же четыре GPU.

Запрос «на третьем этапе 15» здесь трактуется как **ещё 15 эпох после 25**, то есть до 40.
Последняя цель — **65 эпох суммарно на каждый seed**, а не ещё 65 после 40.

| Шаг | Что обучается | Целевая эпоха | Заданий runner | Новых эпох суммарно |
|---|---|---:|---:|---:|
| 1. HPO | 12 конфигураций, seed 42 | 15 | 12 | 180 |
| 2. Seeds | Только победитель HPO, seeds 42 / 123 / 777 | 25 | 3 | 10 + 25 + 25 = 60 |
| 3. Продолжение | Та же конфигурация, те же три seed | 40 | 3 | 15 × 3 = 45 |
| 4. Финальное продолжение | Те же три seed | 65 | 3 | 25 × 3 = 75 |
| **Весь эксперимент** | **12 разных конфигураций** | **15 → 25 → 40 → 65** | **21** | **360** |

21 — число запланированных заданий, **включая продолжения**, не 21 новая конфигурация.
Независимых траекторий обучения — 14: двенадцать на seed 42 и ещё две для победителя.
Все конфигурации HPO сравниваются при одинаковом бюджете 15 эпох. После выбора победителя
повторного HPO нет: остальные 11 конфигураций больше не обучаются.

Используются существующие `run_automated_parameter_search`, SECOND adapter, trainer и
`run_final_benchmark`. У runner три фазы; шаг 3 использует `stage="final"` до 40, а шаг 4 —
документированное расширение той же final-фазы до 65 с постоянным `extension_id`.
Ни нового trainer, ни патчей исходников или подписей backend здесь нет.

**Важно о времени:** 360 — сумма эпох по заданиям, не минуты и не 360 последовательных
эпох на одной GPU. Датасет не урезается. Это ограниченный по числу конфигураций, но уже
не короткий вычислительный эксперимент. Повтор незаписанной работы после аварии может
добавить фактические вычисления сверх планового бюджета.

**Что оцениваем:** классификацию GT/oracle ROI SECOND, не качество каскада с предсказанными
масками. Лучший checkpoint выбирается по validation и может быть раньше 65-й эпохи.
Test запускается отдельно после финального обучения: `MODE="benchmark"` → Run All.


In [1]:
from __future__ import annotations

import copy
import hashlib
import importlib
import importlib.util
import inspect
import json
import math
import os
import time
from datetime import datetime, timezone
from itertools import product
from pathlib import Path
import subprocess
import sys
from collections import Counter
from collections.abc import Mapping

# Укажи путь вручную только при запуске notebook вне репозитория.
PROJECT_ROOT_OVERRIDE = None
DEFAULT_PROJECT_ROOT = Path(
    "/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence"
)
_candidates = (
    [Path(PROJECT_ROOT_OVERRIDE).expanduser()] if PROJECT_ROOT_OVERRIDE else
    [Path.cwd(), *Path.cwd().parents, DEFAULT_PROJECT_ROOT]
)
PROJECT_ROOT = next(
    (p.resolve() for p in _candidates if (p / "src/satchange").is_dir()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден src/satchange. Помести notebook в notebooks/ проекта "
        "или задай PROJECT_ROOT_OVERRIDE."
    )
if not (3, 11) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("Используй Python 3.11/3.12 проекта, как в исходном Tiny notebook.")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
# Тот же исходный код должен быть доступен subprocess/spawn workers.
_pythonpath = [x for x in os.environ.get("PYTHONPATH", "").split(os.pathsep) if x]
if str(SRC) not in _pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join([str(SRC), *_pythonpath])

print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)


Project: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence
Python: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/.venv/bin/python


## 1. Настройки — весь цикл уже включён

Сохранена сетка короткой версии: **3 learning rates × 2 weight decays × 2 dropout = 12**.
Фиксированы batch size 8, `num_workers=4`, BF16 и scheduler `none`.
Четыре GPU получают независимые HPO trials; один trial не распределяется на четыре GPU.

Можно выбрать `FINAL_TARGET = 60`, `65` или `70` **до первого запуска**.
Имя поиска, финальная папка и лимит эпох зависят от этой цели. После начала поиска
не меняй его настройки ради resume. Старые 3-эпоховые и 40-эпоховые searches не импортируются.


In [2]:
MODE = "train_all"  # train_all | check | benchmark

DATA_ROOT = PROJECT_ROOT / "data/SECOND"
CACHE_ROOT = PROJECT_ROOT / "data/SECOND_semantic_cache"
CONFIG_FILE = PROJECT_ROOT / "configs/tiny_siamese_change_classifier.yaml"
PREPARED_MANIFEST = None  # Обычно находится через прежний cache reference.
INSTALL_MISSING = True   # Только небольшие зависимости; НЕ torch/CUDA.

LOCAL_HOLDOUT = True
SPLIT_SEED = 2099
USE_DOCUMENTED_INDEXED_IDS = False
SOURCE_GROUPS = None
SENSOR_METADATA = None
AUDIT_TRAIN_SAMPLES = 16  # Аудит, не ограничение обучения.
SHOW_TRAIN_EXAMPLES = False

GPU_IDS = [0, 1, 2, 3]
PRECISION = "bf16"
NUM_WORKERS = 4
BATCH_SIZE = 8
SCHEDULER = "none"

# У существующего API три фазы. Четвёртый шаг расширяет final.
HPO_TARGET = 15
SEED_TARGET = 25
THIRD_ADDITIONAL_EPOCHS = 15
INTERMEDIATE_TARGET = SEED_TARGET + THIRD_ADDITIONAL_EPOCHS  # 40
FINAL_TARGET = 65  # Допустимы 60 / 65 / 70; выбрать ДО нового поиска.
STAGE_EPOCHS = [HPO_TARGET, SEED_TARGET, INTERMEDIATE_TARGET]
FINAL_EXTENSION_EPOCHS = FINAL_TARGET - INTERMEDIATE_TARGET
FINAL_EXTENSION_ID = f"bounded21_final_{INTERMEDIATE_TARGET}_to_{FINAL_TARGET}_v1"
FINAL_SEEDS = [42, 123, 777]
DEPLOYMENT_SEED = 42  # Зафиксирован заранее, не выбирается по test.

HPO_ROOT = PROJECT_ROOT / "artifacts/hyperparameter_search/semantic/tiny_siamese_second"
SEARCH_NAME = f"second_oracle_bounded21_e15_25_40_{FINAL_TARGET}_{PRECISION}_w{NUM_WORKERS}_v1"
SEARCH_DIRECTORY = HPO_ROOT / SEARCH_NAME
VARIANT = f"tiny_siamese_second_oracle_bounded21_e{FINAL_TARGET}_{PRECISION}_w{NUM_WORKERS}_v1"

HPO_STAGES = [
    ("01_bounded_grid", {
        "learning_rate": [0.0005, 0.001, 0.002],
        "weight_decay": [0.0001, 0.001],
        "model__dropout": [0.0, 0.1],
    }),
]
# При точной ничьей guard остановится; несколько финалистов не запускаются молча.
SELECTION_GAP = 0.0
MAX_HPO_CONFIGURATIONS = 12
MAX_PLANNED_JOBS = MAX_HPO_CONFIGURATIONS + 3 * len(FINAL_SEEDS)
PHASE_TARGETS = {
    "hpo": HPO_TARGET, "seeds": SEED_TARGET,
    "intermediate": INTERMEDIATE_TARGET, "final": FINAL_TARGET,
}
MAX_QUEUED_REMAINING = {
    "hpo": MAX_HPO_CONFIGURATIONS * HPO_TARGET,
    "seeds": (SEED_TARGET - HPO_TARGET) + 2 * SEED_TARGET,
    "intermediate": len(FINAL_SEEDS) * THIRD_ADDITIONAL_EPOCHS,
    "final": len(FINAL_SEEDS) * FINAL_EXTENSION_EPOCHS,
}
MAX_TOTAL_TRAIN_EPOCHS = sum(MAX_QUEUED_REMAINING.values())

EVALUATION_PRECISION = None  # Точность сохранённого training protocol.
BENCHMARK_WARMUP = 20
BENCHMARK_REPETITIONS = 100

if MODE not in {"train_all", "check", "benchmark"}:
    raise ValueError("MODE должен быть train_all, check или benchmark.")
if STAGE_EPOCHS != [15, 25, 40] or FINAL_TARGET not in {60, 65, 70}:
    raise ValueError("Протокол: 15 → 25 → 40 → (60, 65 или 70).")
if FINAL_SEEDS != [42, 123, 777] or DEPLOYMENT_SEED not in FINAL_SEEDS:
    raise ValueError("Нужны seeds [42,123,777] и один из них для deployment.")
if not GPU_IDS or len(GPU_IDS) != len(set(GPU_IDS)) or any(type(x) is not int or x < 0 for x in GPU_IDS):
    raise ValueError("GPU_IDS: уникальные неотрицательные индексы видимых устройств.")
if type(NUM_WORKERS) is not int or NUM_WORKERS < 0 or type(BATCH_SIZE) is not int or BATCH_SIZE < 1:
    raise ValueError("Некорректные NUM_WORKERS/BATCH_SIZE.")
if PRECISION not in {"bf16", "fp32"}:
    raise ValueError("Допустимы bf16 и fp32 с отдельными поисками.")
if len(HPO_STAGES) != 1 or HPO_STAGES[0][0] != "01_bounded_grid":
    raise ValueError("Должен быть один HPO-блок.")
_grid = HPO_STAGES[0][1]
if any(not values or len(set(values)) != len(values) for values in _grid.values()):
    raise ValueError("В сетке пустой список или повторяющиеся значения.")
if math.prod(len(values) for values in _grid.values()) != MAX_HPO_CONFIGURATIONS:
    raise ValueError("Нужны ровно 12 конфигураций.")
if MAX_PLANNED_JOBS != 21:
    raise AssertionError("Неверный лимит заданий.")

print("MODE:", MODE, "| Search:", SEARCH_DIRECTORY)
print("Targets:", list(PHASE_TARGETS.values()))
print(f"Budget: 12 HPO + 3 seeds + 3 intermediate + 3 final = {MAX_PLANNED_JOBS} jobs.")
print("Плановые новые train epochs:", MAX_TOTAL_TRAIN_EPOCHS, MAX_QUEUED_REMAINING)
print("Precision:", PRECISION, "| batch:", BATCH_SIZE, "| workers/job:", NUM_WORKERS)
print("GPU IDs:", GPU_IDS)


MODE: train_all | Search: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/semantic/tiny_siamese_second/second_oracle_bounded21_e15_25_40_65_bf16_w4_v1
Targets: [15, 25, 40, 65]
Budget: 12 HPO + 3 seeds + 3 intermediate + 3 final = 21 jobs.
Плановые новые train epochs: 360 {'hpo': 180, 'seeds': 60, 'intermediate': 45, 'final': 75}
Precision: bf16 | batch: 8 | workers/job: 4
GPU IDs: [0, 1, 2, 3]


## 2. Импорты и проверка установленного API

Используется локальный `src/satchange`, как в исходном notebook. При отсутствии нужного
API код останавливается, а не подставляет бинарный trainer. Проверка отдельного процесса
нужна и для обнаружения проблем импорта PyTorch до запуска очереди.


In [3]:
requirements = {
    "numpy": "numpy", "PIL": "pillow", "cv2": "opencv-python-headless",
    "yaml": "PyYAML", "gdown": "gdown", "matplotlib": "matplotlib",
    "pandas": "pandas", "IPython": "ipython",
}
missing = [pkg for module, pkg in requirements.items() if importlib.util.find_spec(module) is None]
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("В выбранном kernel нет PyTorch. Выбери существующее CUDA-окружение проекта.")
if missing:
    if not INSTALL_MISSING:
        raise RuntimeError(f"Отсутствуют {missing}. Включи INSTALL_MISSING или установи их в kernel.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()

import numpy as np
import pandas as pd
import torch
from IPython.display import display

try:
    import satchange
    from satchange.automated_hpo import run_automated_parameter_search, request_hpo_pause
    from satchange.training import fit_binary_change_model
    from satchange.final_benchmark import run_final_benchmark
    from satchange.models import TinySiameseChangeClassifier
    from satchange.semantic.data import SemanticROIDataset
    from satchange.semantic.second import discover_second, prepare_second
    from satchange.semantic.training import logical_forward, semantic_loss, hpo_parameters
    from satchange.utils import load_config
except ImportError as exc:
    raise RuntimeError(
        "В текущем src/satchange отсутствует часть API исходного Tiny notebook. "
        "Нужны semantic/data.py, semantic/second.py, semantic/training.py, "
        "регистрация Tiny и semantic dispatch в HPO/training/final_benchmark. "
        "Notebook не заменяет эти модули новым trainer. После обновления — Restart Kernel."
    ) from exc


def require_keywords(function, names):
    signature = inspect.signature(function)
    accepts_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in signature.parameters.values())
    absent = set(names) - set(signature.parameters)
    if absent and not accepts_kwargs:
        raise RuntimeError(f"{function.__module__}.{function.__name__} не принимает {sorted(absent)}. Restart Kernel / обнови проект.")


require_keywords(run_automated_parameter_search, {
    "task", "stage_epochs", "stage", "hpo_stage", "selection_gap", "joint_tiebreaker",
    "pending_policy", "dry_run", "seed_patience", "resume_incomplete",
    "additional_epochs", "extension_id",
})
require_keywords(fit_binary_change_model, {"task", "stop_file"})
require_keywords(run_final_benchmark, {"task", "training_results", "checkpoints", "strict_validation"})

_loaded_source = Path(inspect.getfile(TinySiameseChangeClassifier)).resolve()
if not _loaded_source.is_relative_to(SRC.resolve()):
    raise RuntimeError(f"Импортирован не текущий src проекта: {_loaded_source}. Restart Kernel.")
print("Model source:", _loaded_source)
print("Torch:", torch.__version__, "CUDA build:", torch.version.cuda)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<not set>"))
print("Visible GPUs:", torch.cuda.device_count())
_gpu_rows = []
for gpu_id in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(gpu_id)
    _gpu_rows.append({"visible_id": gpu_id, "name": properties.name,
                      "memory_GiB": round(properties.total_memory / 2**30, 2),
                      "selected": gpu_id in GPU_IDS})
if _gpu_rows:
    display(pd.DataFrame(_gpu_rows))
if MODE != "check":
    if not torch.cuda.is_available() or max(GPU_IDS) >= torch.cuda.device_count():
        raise RuntimeError("Не все GPU_IDS доступны в этом kernel. Нумерация относится к видимым CUDA devices.")

# Не обучает модель и не создаёт отдельный smoke search.
_probe_code = (
    "import inspect; from satchange.models import TinySiameseChangeClassifier; "
    "from satchange.semantic.training import hpo_parameters; "
    "from satchange.automated_hpo import run_automated_parameter_search; "
    "print(inspect.getfile(TinySiameseChangeClassifier), flush=True)"
)
try:
    _worker_probe = subprocess.run(
        [sys.executable, "-u", "-c", _probe_code], cwd=PROJECT_ROOT,
        capture_output=True, text=True, timeout=120,
    )
except subprocess.TimeoutExpired as exc:
    raise RuntimeError(
        "Worker import превысил 120 секунд; HPO не запущен. "
        "Проверь отдельный import torch и нагрузку сервера. "
        f"Вывод: {exc.stdout!r}\n{exc.stderr!r}"
    ) from exc
if _worker_probe.returncode:
    raise RuntimeError("Worker import failed:\n" + _worker_probe.stderr)
if str(_loaded_source) not in _worker_probe.stdout:
    raise RuntimeError("Kernel и worker импортируют разные файлы Tiny: " + _worker_probe.stdout)
print("Worker import: OK")
if hasattr(os, "sched_getaffinity"):
    _available_cpus = len(os.sched_getaffinity(0))
    print("CPU affinity:", _available_cpus, "| максимум train workers:", NUM_WORKERS * len(GPU_IDS))
    if NUM_WORKERS * len(GPU_IDS) > _available_cpus:
        print("Внимание: workers больше доступных CPU. Это может не ускорить чтение.")


Model source: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/src/satchange/models/tiny_siamese_change_classifier.py
Torch: 2.12.1+cu130 CUDA build: 13.0
CUDA_VISIBLE_DEVICES: <not set>
Visible GPUs: 4


,visible_id,name,memory_GiB,selected
0,0,NVIDIA A100-SXM4-80GB,79.25,True
1,1,NVIDIA A100-SXM4-80GB,79.25,True
2,2,NVIDIA A100-SXM4-80GB,79.25,True
3,3,NVIDIA A100-SXM4-80GB,79.25,True


Worker import: OK
CPU affinity: 128 | максимум train workers: 16


## 3. Конфигурация и безопасное продолжение

Taxonomy, ROI policy и split берутся из того же YAML и manifest, что в текущем notebook.
В этой версии явно заданы training-настройки ограниченного поиска. Несовместимый YAML или
код под тем же именем не принимаются молча.

Начальные цели API — `[15,25,40]`. После четвёртого шага сохранённая final-цель станет 65.
При повторном Run All она **читается**, а не понижается обратно до 40 и не увеличивается
до 90. Notebook не редактирует `active_phase_targets.json` и подписи HPO вручную.


In [4]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_hash(value):
    normalized = json.loads(json.dumps(value, ensure_ascii=False, default=str))
    return hashlib.sha256(json.dumps(normalized, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def write_json_atomic(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    temporary.replace(path)




PIPELINE_PROTOCOL_PATH = SEARCH_DIRECTORY / "pipeline_protocol.json"
BUDGET_PROTOCOL_PATH = SEARCH_DIRECTORY / "notebook_bounded21_protocol.json"
ACTIVE_TARGETS_PATH = SEARCH_DIRECTORY / "active_phase_targets.json"
pipeline_protocol = read_json(PIPELINE_PROTOCOL_PATH) if PIPELINE_PROTOCOL_PATH.is_file() else None
if pipeline_protocol:
    context = pipeline_protocol.get("context", {})
    if context.get("task") != "semantic_roi_classification" or context.get("model_name") != "tiny_siamese_change_classifier":
        raise ValueError("Сохранённый поиск не относится к Tiny semantic ROI classification.")
    if not BUDGET_PROTOCOL_PATH.is_file():
        raise RuntimeError("У этого SEARCH_NAME уже есть другой pipeline. Выбери новое имя.")
    if canonical_hash(pipeline_protocol["stages"]) != canonical_hash(HPO_STAGES):
        raise RuntimeError("Сохранённая сетка отличается от 12-конфигурационного протокола.")
    if [int(seed) for seed in pipeline_protocol["seeds"]] != FINAL_SEEDS:
        raise RuntimeError("Сохранённые seeds отличаются от настроек.")


def current_stage_epochs():
    """Read runner-owned targets; never rewrite them or reduce an extended final target."""
    targets = read_json(ACTIVE_TARGETS_PATH) if ACTIVE_TARGETS_PATH.is_file() else {}
    values = [targets.get(phase, default) for phase, default in zip(
        ("hpo", "seeds", "final"), STAGE_EPOCHS, strict=True
    )]
    if any(type(value) is not int for value in values):
        raise RuntimeError(f"Некорректные active targets: {values}")
    if values[:2] != [HPO_TARGET, SEED_TARGET] or values[2] not in {INTERMEDIATE_TARGET, FINAL_TARGET}:
        raise RuntimeError(f"Цели поиска {values} выходят за разрешённый протокол. Автоматической миграции нет.")
    return values


print("Saved/current API targets:", current_stage_epochs())
if not CONFIG_FILE.is_file():
    raise FileNotFoundError(f"Нужен существующий YAML Tiny/SECOND: {CONFIG_FILE}")
CONFIG = copy.deepcopy(load_config(CONFIG_FILE))
CONFIG["dropout"] = 0.1
CONFIG.setdefault("training", {}).update({
    "seed": 42, "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
    "precision": PRECISION, "tf32": False,
    "scheduler": SCHEDULER, "learning_rate": 0.001, "weight_decay": 0.0001,
})
_model_check = TinySiameseChangeClassifier(CONFIG["classes"], CONFIG["dropout"])
CLASSES = tuple(_model_check.classes)
PARAMETER_COUNT = sum(p.numel() for p in _model_check.parameters())
print("Class order:", dict(enumerate(CLASSES)))
print("Parameters:", f"{PARAMETER_COUNT:,}", "| parameter tensors FP32 MiB:", round(PARAMETER_COUNT * 4 / 2**20, 3))
print("Это размер параметров, НЕ общая VRAM при обучении.")
del _model_check
print("Effective training config:")
display(CONFIG["training"])


Saved/current API targets: [15, 25, 40]
Class order: {0: 'non-vegetated ground surface', 1: 'tree', 2: 'low vegetation', 3: 'water', 4: 'buildings', 5: 'playgrounds'}
Parameters: 102,118 | parameter tensors FP32 MiB: 0.39
Это размер параметров, НЕ общая VRAM при обучении.
Effective training config:


{'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'batch_size': 8,
 'num_workers': 4,
 'scheduler': 'none',
 'precision': 'bf16',
 'tf32': False,
 'seed': 42,
 'class_weights': None}

## 4. Использовать уже подготовленный SECOND

Сначала используется прежний `notebook_prepared_reference_v2.json` либо manifest сохранённого
текущего ограниченного поиска. Если cache-reference не подходит, вызывается существующий `prepare_second`
для установленного `data/SECOND`. Повторной загрузки и нового распаковщика здесь нет.

Split seed, taxonomy и правила ROI не меняются ради ускорения. Это полный train/validation,
не subset. `AUDIT_TRAIN_SAMPLES=16` ограничивает только просмотр примеров ниже.
Если используется локальный holdout, результаты нельзя называть официальным SECOND test.


In [5]:
# Небольшой указатель позволяет найти подготовленный manifest после Restart Kernel,
# даже когда HPO ещё не запускался. Он не заменяет manifest или pipeline_protocol.
PREPARED_REFERENCE = CACHE_ROOT / "notebook_prepared_reference_v2.json"
_preparation_identity = {
    "data_root": str(DATA_ROOT.resolve()),
    "config": {key: value for key, value in CONFIG.items() if key not in {"training", "dropout"}},
    "split_seed": SPLIT_SEED, "local_holdout": LOCAL_HOLDOUT,
    "indexed_ids": USE_DOCUMENTED_INDEXED_IDS, "source_groups": SOURCE_GROUPS,
    "sensor_metadata": SENSOR_METADATA,
}
_preparation_hash = canonical_hash(_preparation_identity)
PREPARED = Path(PREPARED_MANIFEST).expanduser().resolve() if PREPARED_MANIFEST else None
if pipeline_protocol:
    saved_manifest = Path(pipeline_protocol["dataset_root"]).expanduser().resolve()
    if PREPARED is not None and PREPARED != saved_manifest:
        raise ValueError("PREPARED_MANIFEST отличается от dataset_root сохранённого протокола.")
    PREPARED = saved_manifest
elif PREPARED is None and PREPARED_REFERENCE.is_file():
    reference = read_json(PREPARED_REFERENCE)
    if reference.get("preparation_hash") == _preparation_hash:
        candidate = Path(reference["manifest"])
        if candidate.is_file():
            PREPARED = candidate.resolve()

if PREPARED is None and MODE != "check":
    installed = DATA_ROOT
    try:
        layout = discover_second(installed)
    except (FileNotFoundError, ValueError) as exc:
        raise RuntimeError(
            f"Нет готового manifest и корректного SECOND в {DATA_ROOT}. "
            "Укажи PREPARED_MANIFEST от прежнего запуска либо восстанови установленный датасет. "
            "Повторное скачивание автоматически не запускается."
        ) from exc
    print("Размеченные сцены в поставке:", {key: len(value) for key, value in layout.items()})
    PREPARED = Path(prepare_second(
        installed, CACHE_ROOT, CONFIG,
        split_seed=SPLIT_SEED, local_holdout=LOCAL_HOLDOUT,
        indexed_ids=CONFIG["dataset"]["indexed_id_names"] if USE_DOCUMENTED_INDEXED_IDS else None,
        source_groups=SOURCE_GROUPS, sensor_metadata=SENSOR_METADATA,
    )).resolve()
    if not PREPARED.is_file():
        raise FileNotFoundError(f"prepare_second не вернул существующий manifest: {PREPARED}")
    write_json_atomic(PREPARED_REFERENCE, {"preparation_hash": _preparation_hash, "manifest": str(PREPARED)})

TRAIN_DATASET = None
DATA_DOCUMENT = None
if PREPARED is not None:
    if not PREPARED.is_file():
        raise FileNotFoundError(f"Manifest отсутствует: {PREPARED}. Не создавай новый dataset под старым поиском.")
    DATA_DOCUMENT = read_json(PREPARED)
    TRAIN_DATASET = SemanticROIDataset(PREPARED, "train", CONFIG)
    _verify_start = time.perf_counter()
    TRAIN_DATASET.verify_sources()
    print("Source verification seconds:", round(time.perf_counter() - _verify_start, 2))
    if len(TRAIN_DATASET) == 0:
        raise ValueError("После фильтрации нет training ROI; HPO не запускается.")
    print("Manifest:", PREPARED)
    print("Manifest SHA256:", sha256_file(PREPARED))
    print("Training ROIs:", len(TRAIN_DATASET))
    print("Coverage — отчёт существующего adapter:")
    display(DATA_DOCUMENT.get("coverage", {"notice": "Поле coverage отсутствует; проверь версию adapter."}))
    print("Split provenance:")
    display(DATA_DOCUMENT.get("second", {}).get("local_splits", "Смотри полный provenance в manifest."))
else:
    print("В MODE=check подготовка новых данных не выполняется. Для обучения: MODE=train_all.")


Source verification seconds: 60.17
Manifest: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/SECOND_semantic_cache/0590eed622ef37dcdf469c09be6cd5d83186109d499e755590f1177569d7f7fc/rois/prepared.json
Manifest SHA256: eaa5c557df33a35e2f11039fd36fc1b917e2a1223d7eeaef6eb7582e0c084b79
Training ROIs: 26867
Coverage — отчёт существующего adapter:


{'train': {'proposed': 31196,
  'retained': 26867,
  'exclusions': {'mixed_transition': 4321, 'tied_transition': 8},
  'before_distribution': {'low vegetation': 6382,
   'non-vegetated ground surface': 12299,
   'tree': 2792,
   'buildings': 4947,
   'water': 378,
   'playgrounds': 69},
  'after_distribution': {'non-vegetated ground surface': 9339,
   'low vegetation': 5007,
   'water': 481,
   'tree': 4240,
   'buildings': 7743,
   'playgrounds': 57},
  'transition_distribution': {'low vegetation -> non-vegetated ground surface': 3889,
   'non-vegetated ground surface -> low vegetation': 3539,
   'low vegetation -> water': 275,
   'non-vegetated ground surface -> tree': 2858,
   'low vegetation -> tree': 1034,
   'tree -> non-vegetated ground surface': 1802,
   'non-vegetated ground surface -> water': 167,
   'tree -> low vegetation': 690,
   'buildings -> tree': 292,
   'low vegetation -> buildings': 1178,
   'non-vegetated ground surface -> buildings': 5690,
   'buildings -> non-veg

Split provenance:


['val', 'test']

In [6]:
if TRAIN_DATASET is not None:
    if type(AUDIT_TRAIN_SAMPLES) is not int or AUDIT_TRAIN_SAMPLES < 1:
        raise ValueError("AUDIT_TRAIN_SAMPLES должен быть положительным целым.")
    audit_indices = np.unique(np.linspace(
        0, len(TRAIN_DATASET) - 1, min(AUDIT_TRAIN_SAMPLES, len(TRAIN_DATASET)), dtype=int
    ))
    audit_rows = []
    for index in audit_indices:
        sample = TRAIN_DATASET[int(index)]
        before, after, mask = sample["before"], sample["after"], sample["mask"]
        if before.ndim != 3 or before.shape[0] != 3 or before.shape != after.shape:
            raise ValueError(f"ROI {index}: несовпадающие RGB shapes.")
        height, width = before.shape[-2:]
        if min(height, width) < 4 or mask.shape != (1, height, width):
            raise ValueError(f"ROI {index}: недопустимый размер / mask shape.")
        for name, tensor in (("before", before), ("after", after), ("mask", mask)):
            if not tensor.is_floating_point() or not torch.isfinite(tensor).all() or (tensor < 0).any() or (tensor > 1).any():
                raise ValueError(f"ROI {index}: {name} не является finite [0,1] float tensor.")
        if before.dtype != torch.float32 or after.dtype != torch.float32 or mask.sum() <= 0:
            raise ValueError(f"ROI {index}: нужны float32 RGB и непустая маска.")
        labels = [int(sample["before_label"]), int(sample["after_label"])]
        if any(label < 0 or label >= len(CLASSES) for label in labels):
            raise ValueError(f"ROI {index}: label вне taxonomy: {labels}")
        audit_rows.append({
            "index": int(index), "sample_id": str(sample.get("sample_id", index)),
            "height": height, "width": width, "mask_mass": float(mask.sum()),
            "before_class": CLASSES[labels[0]], "after_class": CLASSES[labels[1]],
        })
    audit_table = pd.DataFrame(audit_rows)
    display(audit_table.head(12))
    print(f"Проверено {len(audit_table)} / {len(TRAIN_DATASET)} train ROI; это выборочный аудит.")
    display(audit_table[["height", "width", "mask_mass"]].describe())
    print("Переходы В ПРОВЕРЕННОЙ ВЫБОРКЕ, не распределение всего SECOND:")
    display(audit_table.groupby(["before_class", "after_class"]).size().rename("n_sampled").reset_index())
else:
    print("Аудит реальных ROI пропущен: manifest ещё не выбран.")


,index,sample_id,height,width,mask_mass,before_class,after_class
0,0,train/00003,361,319,4005.0,low vegetation,non-vegetated ground surface
1,1791,train/00425,128,128,2.0,low vegetation,non-vegetated ground surface
2,3582,train/00817,128,128,59.0,tree,low vegetation
3,5373,train/01154,128,128,519.0,non-vegetated ground surface,tree
4,7164,train/01596,128,128,54.0,non-vegetated ground surface,tree
5,8955,train/01912,245,158,19878.0,buildings,non-vegetated ground surface
6,10746,train/02528,128,128,97.0,non-vegetated ground surface,low vegetation
7,12537,train/03113,128,128,2406.0,non-vegetated ground surface,low vegetation
8,14328,train/03596,149,166,9232.0,non-vegetated ground surface,buildings
9,16119,train/04486,203,128,1556.0,non-vegetated ground surface,tree


Проверено 16 / 26867 train ROI; это выборочный аудит.


,height,width,mask_mass
count,16.000000,16.000000,16.000000
mean,156.062500,146.250000,2804.812500
std,64.006738,48.017358,5112.611377
min,128.000000,128.000000,2.000000
25%,128.000000,128.000000,166.750000
50%,128.000000,128.000000,1035.500000
75%,135.500000,135.500000,2457.000000
max,361.000000,319.000000,19878.000000


Переходы В ПРОВЕРЕННОЙ ВЫБОРКЕ, не распределение всего SECOND:


,before_class,after_class,n_sampled
0,buildings,non-vegetated ground surface,3
1,low vegetation,non-vegetated ground surface,3
2,non-vegetated ground surface,buildings,3
3,non-vegetated ground surface,low vegetation,2
4,non-vegetated ground surface,tree,3
5,tree,low vegetation,2


## 5. Проверка модели — без эпох на SECOND

Синтетический forward/backward на маленьких tensors и проверка BF16 на первой выбранной GPU.
Это **не** реальный trainer/resume smoke и не отдельные HPO-trials. Работа переходов resume
проверяется далее непосредственно на выбранной конфигурации; никакой fake smoke-report не создаётся.

При AMP изображения остаются `float32`, как требует предоставленная CNN; `autocast` применяется
к forward/loss, а не через `model.bfloat16()`. Сам training использует `precision` существующего
backend; синтетическая проверка подтверждает только совместимость CNN с BF16.


In [7]:
_previous_threads = torch.get_num_threads()
_cpu_rng = torch.get_rng_state()
torch.set_num_threads(1)
try:
    model = TinySiameseChangeClassifier(CLASSES, CONFIG["dropout"])
    conv_shapes = [(layer.in_channels, layer.out_channels) for layer in model.backbone if isinstance(layer, torch.nn.Conv2d)]
    if conv_shapes != [(3, 32), (32, 64), (64, 128)]:
        raise AssertionError(f"Импортирована другая backbone-архитектура: {conv_shapes}")
    samples = [
        {"before": torch.rand(3, h, w), "after": torch.rand(3, h, w),
         "mask": torch.ones(1, h, w), "before_label": 0, "after_label": 1}
        for h, w in [(17, 23), (19, 29)]
    ]
    logits, before_labels, after_labels = logical_forward(model, samples, torch.device("cpu"))
    assert set(logits) == {"before_logits", "after_logits"}
    assert all(t.shape == (2, len(CLASSES)) for t in logits.values())
    loss, _ = semantic_loss(logits, before_labels, after_labels)
    loss.backward()
    assert torch.isfinite(loss)
    assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())

    model.eval()
    with torch.inference_mode():
        for h, w in [(4, 4), (5, 7), (17, 23)]:
            before, after = torch.rand(1, 3, h, w), torch.rand(1, 3, h, w)
            mask = torch.zeros(1, 1, h, w)
            mask[..., -1, -1] = 1  # Проверяем и край нечётного native grid.
            out = model(before, after, mask)
            swapped = model(after, before, mask)
            same = model(before, before, mask)
            assert all(torch.isfinite(x).all() for x in out.values())
            torch.testing.assert_close(out["before_logits"], swapped["after_logits"])
            torch.testing.assert_close(same["before_logits"], same["after_logits"])
        try:
            model(before, after, torch.zeros_like(mask))
        except ValueError:
            pass
        else:
            raise AssertionError("Пустая маска должна быть отклонена.")
    print("Synthetic implementation smoke: PASS")
    print("Output shapes:", {key: tuple(value.shape) for key, value in logits.items()})
    print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
finally:
    torch.set_rng_state(_cpu_rng)
    torch.set_num_threads(_previous_threads)
    for _name in ("model", "logits", "loss", "samples", "out", "swapped", "same"):
        globals().pop(_name, None)


Synthetic implementation smoke: PASS
Output shapes: {'before_logits': (2, 6), 'after_logits': (2, 6)}
Parameters: 102,118


In [8]:
if MODE == "train_all":
    if PRECISION == "bf16":
        for gpu_id in GPU_IDS:
            with torch.cuda.device(gpu_id):
                if not torch.cuda.is_bf16_supported():
                    raise RuntimeError(f"cuda:{gpu_id} не поддерживает BF16 в текущем окружении.")
    _device = torch.device(f"cuda:{GPU_IDS[0]}")
    with torch.random.fork_rng(devices=[GPU_IDS[0]]):
        _model = TinySiameseChangeClassifier(CLASSES, CONFIG["dropout"]).to(_device)
        _samples = [
            {"before": torch.rand(3, h, w), "after": torch.rand(3, h, w),
             "mask": torch.ones(1, h, w), "before_label": 0, "after_label": 1}
            for h, w in [(17, 23), (19, 29)]
        ]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=PRECISION == "bf16"):
            _outputs, _before_labels, _after_labels = logical_forward(_model, _samples, _device)
            _loss, _ = semantic_loss(_outputs, _before_labels, _after_labels)
        _loss.backward()
        if not torch.isfinite(_loss).item() or any(
            p.grad is not None and not torch.isfinite(p.grad).all().item() for p in _model.parameters()
        ):
            raise RuntimeError("Synthetic GPU forward/backward returned nonfinite values.")
        if PRECISION == "bf16" and any(value.dtype != torch.bfloat16 for value in _outputs.values()):
            raise RuntimeError("Synthetic logits не BF16: проверь logical_forward/autocast установленной версии.")
        torch.cuda.synchronize(_device)
        print("Synthetic GPU forward/backward: PASS; logits:", {k: str(v.dtype) for k, v in _outputs.items()})
        del _model, _samples, _outputs, _loss, _before_labels, _after_labels
else:
    print("GPU preflight не выполняется в MODE=", MODE)


Synthetic GPU forward/backward: PASS; logits: {'before_logits': 'torch.bfloat16', 'after_logits': 'torch.bfloat16'}


## 6. OPTIONS — 12 HPO-конфигураций и один победитель

Сохранён `selection_score` исходного notebook: среднее per-date macro F1. Это не F1
пространственной маски. Все 12 конфигураций получают по 15 эпох. При `selection_gap=0`
в seed-фазу должен перейти один победитель. При точной ничьей notebook остановится до
расширения очереди: лимит не обходится.

Checkpoint interval — одна эпоха; early stopping отключён. Отдельного многочасового smoke
на всём SECOND нет. Переходы продолжают собственный `last.pt` каждого seed. Финальные
результаты берутся из `best.pt` по validation, а не обязательно из последней эпохи.

Промежуточная final-фаза до 40 и её расширение до 65 используют одни гиперпараметры.
Промежуточные validation-отчёты сохраняются, но не используются для нового подбора сетки.


In [9]:
BASE = hpo_parameters(CONFIG, PREPARED) if PREPARED is not None else None
if MODE != "check" and BASE is None:
    raise FileNotFoundError("Подготовленный manifest отсутствует. HPO без реальных данных запрещён.")
if BASE is not None:
    for key, expected in {
        "seed": 42, "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
        "precision": PRECISION, "scheduler": SCHEDULER,
    }.items():
        if BASE.get(key) != expected:
            raise RuntimeError(f"hpo_parameters вернул {key}={BASE.get(key)!r}, ожидается {expected!r}.")
    if tuple(BASE["semantic_config"]["classes"]) != CLASSES:
        raise RuntimeError("Taxonomy BASE не совпадает с CONFIG.")
    if {"image_size", "pos_weight", "bce_weight", "dice_weight"}.intersection(BASE):
        raise RuntimeError("В semantic BASE попали binary-specific параметры.")
    if pipeline_protocol:
        saved_base = pipeline_protocol["base"]
        changed = [key for key in BASE if key not in saved_base or canonical_hash(BASE[key]) != canonical_hash(saved_base[key])]
        if changed:
            raise RuntimeError(f"BASE изменился под тем же SEARCH_NAME: {changed}. Нужен отдельный новый поиск.")

OPTIONS = dict(
    task="semantic_roi_classification", stages=HPO_STAGES,
    base_parameters=BASE, model_name="tiny_siamese_change_classifier", dataset_root=PREPARED,
    stage_epochs=STAGE_EPOCHS, seeds=FINAL_SEEDS, additional_seeds=True,
    output_root=HPO_ROOT, search_name=SEARCH_NAME, gpu_ids=GPU_IDS,
    reuse_from=[], resume_from=[], resume_incomplete=True, allow_legacy_import=False,
    patience=None, min_epochs=1, min_delta=0.0, seed_patience=None,
    selection_gap=SELECTION_GAP, joint_tiebreaker=False,
    checkpoint_interval=1, max_trial_memory_mb=80_000,
    oom_retry_delay_seconds=10, max_oom_attempts=1,
)
inspect.signature(run_automated_parameter_search).bind_partial(**OPTIONS, stage="hpo")
_source_paths = sorted({
    *list((SRC / "satchange/semantic").glob("*.py")),
    *[SRC / "satchange" / name for name in (
        "automated_hpo.py", "hpo_core.py", "hpo_backend.py", "training.py", "final_benchmark.py"
    )], Path(inspect.getfile(TinySiameseChangeClassifier)),
})
SOURCE_HASHES = {str(path.relative_to(SRC)): sha256_file(path) for path in _source_paths if path.is_file()}
BUDGET_IDENTITY = {
    "notebook_protocol": "bounded21_v1", "stages": HPO_STAGES,
    "initial_stage_epochs": STAGE_EPOCHS, "phase_targets": PHASE_TARGETS,
    "final_extension_id": FINAL_EXTENSION_ID,
    "final_extension_epochs": FINAL_EXTENSION_EPOCHS,
    "base": BASE, "seeds": FINAL_SEEDS, "selection_gap": SELECTION_GAP,
    "dataset_root": str(PREPARED), "manifest_sha256": sha256_file(PREPARED) if PREPARED else None,
    "source_hashes": SOURCE_HASHES, "torch": str(torch.__version__),
    "max_planned_jobs": MAX_PLANNED_JOBS, "max_total_train_epochs": MAX_TOTAL_TRAIN_EPOCHS,
}
BUDGET_FINGERPRINT = canonical_hash(BUDGET_IDENTITY)
if BUDGET_PROTOCOL_PATH.is_file():
    previous = read_json(BUDGET_PROTOCOL_PATH)
    if previous.get("fingerprint") != BUDGET_FINGERPRINT:
        raise RuntimeError("Ограниченный protocol изменился. Не подменяй подписи: используй новое SEARCH_NAME для нового эксперимента.")
elif MODE != "check":
    write_json_atomic(BUDGET_PROTOCOL_PATH, {"fingerprint": BUDGET_FINGERPRINT, "identity": BUDGET_IDENTITY})

GRID_KEYS = tuple(HPO_STAGES[0][1])
EXPECTED_GRID = {
    tuple(combination) for combination in product(*(HPO_STAGES[0][1][key] for key in GRID_KEYS))
}
print("Фактическая HPO-сетка:")
display(pd.DataFrame([dict(zip(GRID_KEYS, values)) for values in sorted(EXPECTED_GRID)]))
print("Absolute targets:", STAGE_EPOCHS, "| All data: no subset/max_samples override.")
print("Fixed batch / workers / precision:", BATCH_SIZE, NUM_WORKERS, PRECISION)
print(f"Предел: {MAX_PLANNED_JOBS} jobs / {MAX_TOTAL_TRAIN_EPOCHS} плановых train epochs с нуля.")
print("Четыре шага:", PHASE_TARGETS, "| extension:", FINAL_EXTENSION_ID)


Фактическая HPO-сетка:


,learning_rate,weight_decay,model__dropout
0,0.0005,0.0001,0.0
1,0.0005,0.0001,0.1
2,0.0005,0.0010,0.0
3,0.0005,0.0010,0.1
4,0.0010,0.0001,0.0
5,0.0010,0.0001,0.1
6,0.0010,0.0010,0.0
7,0.0010,0.0010,0.1
8,0.0020,0.0001,0.0
9,0.0020,0.0001,0.1


Absolute targets: [15, 25, 40] | All data: no subset/max_samples override.
Fixed batch / workers / precision: 8 4 bf16
Предел: 21 jobs / 360 плановых train epochs с нуля.
Четыре шага: {'hpo': 15, 'seeds': 25, 'intermediate': 40, 'final': 65} | extension: bounded21_final_40_to_65_v1


## 7. Проверки бюджета и этапов

Каждому обучению предшествует проверка `dry_run`: число jobs, seeds, target, оставшиеся
эпохи, неизменность batch/workers/precision и одна конфигурация после HPO.

Служебные JSON сохраняют результаты законченных шагов. При resume notebook не требует
заново выполнить HPO или промежуточные 40 эпох после старта финального продолжения.
`final_choice.json` и код выбора победителя принадлежат существующему runner.

Снимок отчёта на 40-й эпохе сохраняет метрики и ссылки, **не копирует веса**. Ссылки на
checkpoints не следует считать архивной копией состояния на 40, если backend меняет
тот же файл при продолжении.


In [10]:
def show_plan(plan, title):
    rows = []
    for item in plan["jobs"]:
        source = item.get("source") or {}
        saved_epoch = int(source.get("epoch", 0))
        target = int(item["target_epoch"])
        reused = bool(item.get("reused", False))
        remaining = 0 if reused else target - saved_epoch
        if remaining < 0:
            raise RuntimeError("План пытается продолжить checkpoint с эпохи выше target.")
        parameters = item["parameters"]
        rows.append({
            "seed": int(parameters["seed"]), "saved_epoch": saved_epoch,
            "target_epoch": target, "additional_epochs": remaining,
            "action": "reuse" if reused else "resume" if saved_epoch else "fresh",
            "lr": parameters.get("learning_rate"), "batch": parameters.get("batch_size"),
            "scheduler": parameters.get("scheduler"),
            "dropout": parameters.get("model_parameters", {}).get("dropout"),
        })
    frame = pd.DataFrame(rows)
    print(title)
    display(frame)
    print("Retained:", plan.get("retained_trials", len(rows)), "Skipped:", plan.get("skipped_trials", 0))
    print("Remaining epochs summed over jobs:", sum(row["additional_epochs"] for row in rows))
    print("Plan:", plan.get("plan_path"))
    if not rows:
        raise RuntimeError("Очередь пуста. Для нового поиска нужна policy='all'; finish_started не создаёт новые seed-runs.")
    return frame


def require_complete(summary, label):
    if not summary.get("complete", False):
        raise RuntimeError(f"{label} не завершён. Проверь errors/results ниже и повтори этот этап, не переходи к следующему.")


def show_seed_ranking(summary):
    ranking = summary.get("ranking", [])
    rows = []
    for position, item in enumerate(ranking, 1):
        if "mean_selection_score" not in item or "std_selection_score" not in item:
            raise RuntimeError("Ожидался semantic ranking с mean/std_selection_score, а не binary F1/IoU ranking.")
        if {int(seed) for seed in item["seeds"]} != set(FINAL_SEEDS):
            raise RuntimeError("В ranking оказалась неполная группа seeds.")
        rows.append({
            "rank": position, "seeds": item["seeds"],
            "mean_macro_F1_two_dates": item["mean_selection_score"],
            "sample_std_macro_F1_two_dates": item["std_selection_score"],
            "parameters": item["parameters"],
        })
    display(pd.DataFrame(rows))
    print("Incomplete families (не ранжируются):", len(summary.get("incomplete_families", [])))
    print("Selection scope:", summary.get("selection_scope", "см. summary"))
    return rows


def checked_final(summary, target):
    require_complete(summary, "Final")
    checkpoints = {int(seed): Path(path) for seed, path in summary["best_checkpoints"].items()}
    if set(checkpoints) != set(FINAL_SEEDS) or len(summary.get("results", [])) != len(FINAL_SEEDS):
        raise RuntimeError("Нужны три финальных результата и три seed checkpoints.")
    result_seeds = set()
    identities = []
    for row in summary["results"]:
        if int(row["epoch"]) != int(target):
            raise RuntimeError(f"Final run не достиг абсолютного target={target}.")
        params = row["parameters"]
        validate_parameters(params, "final result")
        result_seeds.add(int(params["seed"]))
        if tuple(params["semantic_config"]["classes"]) != CLASSES:
            raise RuntimeError("В финальные results попала другая taxonomy.")
        # Не сравниваем служебные поля пути/бюджета; научные параметры одинаковы.
        scientific_keys = scientific_parameter_keys()
        missing_keys = scientific_keys - set(params)
        if missing_keys:
            raise RuntimeError(f"В final parameters отсутствуют поля BASE: {sorted(missing_keys)}")
        identities.append({key: params[key] for key in scientific_keys})
    if result_seeds != set(FINAL_SEEDS) or len({canonical_hash(identity) for identity in identities}) != 1:
        raise RuntimeError("Финальные seeds не независимы по ID или используют разные гиперпараметры.")
    if len({path.resolve() for path in checkpoints.values()}) != len(FINAL_SEEDS):
        raise RuntimeError("Нескольким seeds назначен один и тот же checkpoint.")
    for seed, path in sorted(checkpoints.items()):
        if not path.is_file():
            raise FileNotFoundError(path)
        last_path = path.parent / "last.pt"
        if not last_path.is_file():
            raise FileNotFoundError(f"Нет last.pt для seed={seed}: {last_path}")
        print(f"seed={seed}: {path}")
    return checkpoints





def scientific_parameter_keys():
    return set(BASE) - {"seed", "budget_epochs", "dataset_root", "output_dir", "run_directory"}


def parameter_value(parameters, key):
    if key.startswith("model__"):
        return parameters["model_parameters"][key.split("__", 1)[1]]
    return parameters[key]


def parameter_family(parameters):
    missing = scientific_parameter_keys() - set(parameters)
    if missing:
        raise RuntimeError(f"В parameters отсутствуют поля BASE: {sorted(missing)}")
    return canonical_hash({key: parameters[key] for key in scientific_parameter_keys()})


def validate_parameters(parameters, label):
    """Validate the fixed base and the explicit grid, without changing either."""
    values = tuple(parameter_value(parameters, key) for key in GRID_KEYS)
    if values not in EXPECTED_GRID:
        raise RuntimeError(f"{label}: конфигурация вне разрешённой HPO-сетки: {values}")
    expected = copy.deepcopy(BASE)
    for key, value in zip(GRID_KEYS, values, strict=True):
        if key.startswith("model__"):
            expected["model_parameters"][key.split("__", 1)[1]] = value
        else:
            expected[key] = value
    changed = [key for key in scientific_parameter_keys()
               if key not in parameters or canonical_hash(parameters[key]) != canonical_hash(expected[key])]
    if changed:
        raise RuntimeError(f"{label}: изменились фиксированные параметры: {sorted(changed)}")
    return values


def completed_target(summary):
    """Return one completed final target; an incomplete summary is never completion."""
    if not summary or not summary.get("complete", False):
        return None
    results = summary.get("results", [])
    if len(results) != len(FINAL_SEEDS):
        raise RuntimeError("Complete final summary должен содержать три seed-результата.")
    seeds = [int(row["parameters"]["seed"]) for row in results]
    if sorted(seeds) != sorted(FINAL_SEEDS):
        raise RuntimeError("Некорректные/повторяющиеся seeds в complete final summary.")
    targets = {int(row["epoch"]) for row in results}
    if len(targets) != 1:
        raise RuntimeError("Complete final summary содержит разные достигнутые эпохи.")
    return next(iter(targets))


PHASE_RECORD_DIRECTORY = SEARCH_DIRECTORY / "notebook_phase_results"
RUNTIME_REPORT_PATH = SEARCH_DIRECTORY / "notebook_bounded21_runtime.json"


def phase_record_path(phase):
    if phase not in PHASE_TARGETS:
        raise ValueError(f"Неизвестный шаг: {phase}")
    return PHASE_RECORD_DIRECTORY / f"{phase}_e{PHASE_TARGETS[phase]}.json"


def validate_phase_result(summary, phase):
    require_complete(summary, phase)
    if phase == "hpo":
        if "survivors" not in summary or len(summary["survivors"]) != 1:
            count = len(summary.get("survivors", []))
            raise RuntimeError(
                f"После HPO осталось {count} survivors вместо одного. "
                "Возможна точная ничья при selection_gap=0. Лимит 21 job не расширяется."
            )
    elif phase == "seeds":
        ranking = summary.get("ranking", [])
        if len(ranking) != 1 or sorted(int(seed) for seed in ranking[0]["seeds"]) != sorted(FINAL_SEEDS):
            raise RuntimeError("Seed ranking должен содержать одну полную конфигурацию на трёх seeds.")
        if "mean_selection_score" not in ranking[0] or "std_selection_score" not in ranking[0]:
            raise RuntimeError("Ожидался semantic ranking с mean/std_selection_score.")
    elif phase in {"intermediate", "final"}:
        checked_final(summary, target=PHASE_TARGETS[phase])
    else:
        raise ValueError(phase)


def read_phase_record(phase):
    path = phase_record_path(phase)
    if not path.is_file():
        return None
    record = read_json(path)
    if (record.get("fingerprint") != BUDGET_FINGERPRINT
            or record.get("phase") != phase
            or record.get("target") != PHASE_TARGETS[phase]):
        raise RuntimeError(f"Служебный отчёт другого протокола: {path}")
    if record.get("summary_sha256") != canonical_hash(record.get("summary")):
        raise RuntimeError(f"Служебный отчёт изменён/неполон: {path}")
    validate_phase_result(record["summary"], phase)
    return record


def save_phase_record(phase, summary, approved):
    # Only metadata are saved here. Checkpoint state belongs to the existing runner.
    validate_phase_result(summary, phase)
    record = {
        "fingerprint": BUDGET_FINGERPRINT, "phase": phase,
        "target": PHASE_TARGETS[phase], "approved_plan": approved,
        "summary": summary, "summary_sha256": canonical_hash(summary),
        "saved_utc": datetime.now(timezone.utc).isoformat(),
        "note": "Metadata snapshot; checkpoint paths are references, not archived weight copies.",
    }
    write_json_atomic(phase_record_path(phase), record)
    return record


def validate_bounded_plan(plan, phase):
    """Reject expanded or incompatible plans before the existing runner trains them."""
    if phase not in PHASE_TARGETS:
        raise ValueError(phase)
    jobs = plan.get("jobs", [])
    expected_count = MAX_HPO_CONFIGURATIONS if phase == "hpo" else len(FINAL_SEEDS)
    if len(jobs) != expected_count:
        raise RuntimeError(
            f"{phase}: запланировано {len(jobs)}, разрешено {expected_count}. "
            "Бюджет не расширяется; проверь survivors / точную ничью."
        )
    target = PHASE_TARGETS[phase]
    seen_seeds, grid_rows, families = [], [], set()
    remaining = 0
    for job in jobs:
        parameters = job["parameters"]
        seed = int(parameters["seed"])
        seen_seeds.append(seed)
        if int(job["target_epoch"]) != target:
            raise RuntimeError(f"{phase}: target не равен фиксированному {target}.")
        source = job.get("source") or {}
        saved = int(source.get("epoch", 0))
        reused = bool(job.get("reused", False))
        if saved < 0 or saved > target:
            raise RuntimeError(f"{phase}: нельзя использовать эпоху {saved} для target={target}.")
        source_parameters = source.get("parameters")
        source_seed = source.get("seed")
        if isinstance(source_parameters, dict):
            source_seed = source_parameters.get("seed", source_seed)
        if source_seed is not None and int(source_seed) != seed:
            raise RuntimeError(f"{phase}: checkpoint другого seed ({source_seed} вместо {seed}).")
        if not reused:
            remaining += target - saved
            if phase == "seeds" and seed == 42 and saved < HPO_TARGET:
                raise RuntimeError("Seed 42 должен продолжаться из HPO, а не начинаться заново.")
            if phase == "intermediate" and saved < SEED_TARGET:
                raise RuntimeError("Каждый seed должен продолжаться с собственной эпохи >=25.")
            if phase == "final" and saved < INTERMEDIATE_TARGET:
                raise RuntimeError("Финальное продолжение требует собственный last-state >=40 для каждого seed.")
        values = validate_parameters(parameters, phase)
        grid_rows.append(values)
        families.add(parameter_family(parameters))
    if phase == "hpo":
        if set(seen_seeds) != {42} or set(grid_rows) != EXPECTED_GRID or len(set(grid_rows)) != len(jobs):
            raise RuntimeError("HPO: нужны ровно 12 разных конфигураций на seed 42.")
    elif sorted(seen_seeds) != sorted(FINAL_SEEDS) or len(families) != 1:
        raise RuntimeError(f"{phase}: нужны ровно три разных seed одной конфигурации.")
    if phase == "final":
        intermediate = read_phase_record("intermediate")
        if intermediate is None:
            raise RuntimeError("Нет подтверждённого промежуточного результата до 40 эпох.")
        previous_family = parameter_family(intermediate["summary"]["results"][0]["parameters"])
        if families != {previous_family}:
            raise RuntimeError("При 40→final изменилась выбранная конфигурация.")
    if remaining > MAX_QUEUED_REMAINING[phase]:
        raise RuntimeError(f"{phase}: {remaining} оставшихся эпох превышают лимит {MAX_QUEUED_REMAINING[phase]}.")
    return {"phase": phase, "jobs": len(jobs), "target": target, "remaining_epochs": remaining}


def phase_call(phase):
    options = copy.deepcopy(OPTIONS)
    active = current_stage_epochs()
    options["stage_epochs"] = active
    if phase == "hpo":
        call = {"stage": "hpo", "hpo_stage": HPO_STAGES[0][0]}
    elif phase == "seeds":
        call = {"stage": "seeds", "pending_policy": "all"}
    elif phase == "intermediate":
        if active[2] != INTERMEDIATE_TARGET:
            raise RuntimeError("Нельзя вернуть final с сохранённой конечной цели обратно к 40.")
        call = {"stage": "final"}
    elif phase == "final":
        if read_phase_record("intermediate") is None:
            raise RuntimeError("Сначала должны завершиться все три seed до 40 эпох.")
        call = {"stage": "final"}
        if active[2] == INTERMEDIATE_TARGET:
            call.update(additional_epochs=FINAL_EXTENSION_EPOCHS, extension_id=FINAL_EXTENSION_ID)
        # If extension already persisted FINAL_TARGET, resume it without another addition.
    else:
        raise ValueError(phase)
    inspect.signature(run_automated_parameter_search).bind_partial(**options, **call)
    return options, call


def run_guarded_phase(phase):
    cached = read_phase_record(phase)
    if cached is not None:
        print(f"{phase}: готовый отчёт до {PHASE_TARGETS[phase]} прочитан; обучение не вызывается.")
        return cached["summary"]
    if phase != "final" and current_stage_epochs()[2] == FINAL_TARGET:
        raise RuntimeError(
            f"Final уже расширен, но служебный отчёт {phase} отсутствует. "
            "Не начинаем прошлые этапы заново под уменьшенной целью."
        )
    options, call = phase_call(phase)
    plan = run_automated_parameter_search(**options, **call, dry_run=True)
    approved = validate_bounded_plan(plan, phase)
    show_plan(plan, f"{phase.upper()} — проверенный план")
    print("Budget guard: PASS;", approved)
    # dry_run may persist an extension target. Re-read it before the actual invocation.
    options, call = phase_call(phase)
    started = time.perf_counter()
    status = "running"
    try:
        result = run_automated_parameter_search(**options, **call)
        validate_phase_result(result, phase)
        save_phase_record(phase, result, approved)
        status = "complete"
        return result
    except BaseException:
        status = "interrupted_or_failed"
        raise
    finally:
        elapsed = time.perf_counter() - started
        try:
            record = read_json(RUNTIME_REPORT_PATH) if RUNTIME_REPORT_PATH.is_file() else {"calls": []}
            if record.get("fingerprint", BUDGET_FINGERPRINT) != BUDGET_FINGERPRINT:
                raise RuntimeError("Runtime report принадлежит другому протоколу.")
            record["fingerprint"] = BUDGET_FINGERPRINT
            record["calls"].append({
                **approved, "status": status, "wall_seconds": elapsed,
                "finished_utc": datetime.now(timezone.utc).isoformat(),
            })
            write_json_atomic(RUNTIME_REPORT_PATH, record)
        except Exception as log_error:
            # Do not replace an actual training exception with a logging exception.
            print("Не удалось записать вспомогательный runtime report:", repr(log_error))
        print(f"{phase}: wall time {elapsed / 60:.2f} min; status={status}")


## 8. Один запуск: HPO → seeds → +15 эпох → final до 65

`train_all` выполняет все четыре шага без ручного переключения режима.
Завершённые шаги читаются из служебных отчётов, незавершённые — продолжаются существующим
runner. До конца 65-й эпохи benchmark не разрешается.

Четвёртый шаг один раз расширяет final-цель на 25 эпох с постоянным `extension_id`.
После этого обычный resume использует сохранённый target=65 **без нового прибавления**.
Контроллер защищён файловым lock от второго запуска этого же notebook для того же поиска.


In [11]:
FINAL_SUMMARY_PATH = SEARCH_DIRECTORY / "final_training_summary.json"
SEED_SUMMARY_PATH = SEARCH_DIRECTORY / "seed_validation_summary.json"
FINAL_CHECKPOINTS = None
final_summary = read_json(FINAL_SUMMARY_PATH) if FINAL_SUMMARY_PATH.is_file() else None

if MODE == "train_all":
    import fcntl
    SEARCH_DIRECTORY.mkdir(parents=True, exist_ok=True)
    with (SEARCH_DIRECTORY / "notebook_bounded21_controller.lock").open("a+") as _lock:
        try:
            fcntl.flock(_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError("Этим поиском уже управляет другой controller.") from exc
        try:
            # Read live runner state only after taking the lock.
            final_summary = read_json(FINAL_SUMMARY_PATH) if FINAL_SUMMARY_PATH.is_file() else None
            reached = completed_target(final_summary)
            if reached is not None and reached not in {INTERMEDIATE_TARGET, FINAL_TARGET}:
                raise RuntimeError(f"В поиске другой завершённый target={reached}.")
            if reached == FINAL_TARGET:
                FINAL_CHECKPOINTS = checked_final(final_summary, target=FINAL_TARGET)
                if read_phase_record("final") is None:
                    save_phase_record("final", final_summary, {
                        "phase": "final", "jobs": 3, "target": FINAL_TARGET,
                        "remaining_epochs": 0, "recovered_from_runner_summary": True,
                    })
                print("Все три seed уже достигли конечной цели. Новых эпох не добавляем.")
            else:
                # Recover metadata if interruption occurred after final@40 finished
                # but before its notebook-side snapshot was saved.
                if reached == INTERMEDIATE_TARGET and read_phase_record("intermediate") is None:
                    save_phase_record("intermediate", final_summary, {
                        "phase": "intermediate", "jobs": 3, "target": INTERMEDIATE_TARGET,
                        "remaining_epochs": 0, "recovered_from_runner_summary": True,
                    })
                intermediate_record = read_phase_record("intermediate")
                if intermediate_record is None:
                    hpo_summary = run_guarded_phase("hpo")
                    print("Один победитель HPO; остальные 11 конфигураций больше не обучаются.")
                    seed_summary = run_guarded_phase("seeds")
                    show_seed_ranking(seed_summary)
                    intermediate_summary = run_guarded_phase("intermediate")
                    print(f"Промежуточное обучение до {INTERMEDIATE_TARGET} завершено.")
                else:
                    print("Промежуточные 40 эпох уже завершены; HPO / seeds не перезапускаются.")
                final_summary = run_guarded_phase("final")
                FINAL_CHECKPOINTS = checked_final(final_summary, target=FINAL_TARGET)
        finally:
            fcntl.flock(_lock, fcntl.LOCK_UN)
elif completed_target(final_summary) == FINAL_TARGET:
    FINAL_CHECKPOINTS = checked_final(final_summary, target=FINAL_TARGET)
else:
    print("Обучение не вызывается в MODE=", MODE)
    if final_summary:
        print("Текущий final summary ещё не подтверждает конечный бюджет для всех трёх seeds.")

if FINAL_CHECKPOINTS:
    print(f"\nTRAINING COMPLETE — target={FINAL_TARGET} на каждом seed")
    print("Final summary:", FINAL_SUMMARY_PATH)
    print("Deployment seed:", DEPLOYMENT_SEED)
    choice_path = SEARCH_DIRECTORY / "final_choice.json"
    if choice_path.is_file():
        display(read_json(choice_path))
    print("Test не используется для выбора. MODE='benchmark' запускает отдельную оценку.")


01_bounded_grid: target=15, retained=12, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/semantic/tiny_siamese_second/second_oracle_bounded21_e15_25_40_65_bf16_w4_v1/operations/01_bounded_grid_e15_4522bbcf/plan.json
HPO — проверенный план


,seed,saved_epoch,target_epoch,additional_epochs,action,lr,batch,scheduler,dropout
0,42,0,15,15,fresh,0.0005,8,none,0.0
1,42,0,15,15,fresh,0.0005,8,none,0.1
2,42,0,15,15,fresh,0.0005,8,none,0.0
3,42,0,15,15,fresh,0.0005,8,none,0.1
4,42,0,15,15,fresh,0.0010,8,none,0.0
5,42,0,15,15,fresh,0.0010,8,none,0.1
6,42,0,15,15,fresh,0.0010,8,none,0.0
7,42,0,15,15,fresh,0.0010,8,none,0.1
8,42,0,15,15,fresh,0.0020,8,none,0.0
9,42,0,15,15,fresh,0.0020,8,none,0.1


Retained: 12 Skipped: 0
Remaining epochs summed over jobs: 180
Plan: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/semantic/tiny_siamese_second/second_oracle_bounded21_e15_25_40_65_bf16_w4_v1/operations/01_bounded_grid_e15_4522bbcf/plan.json
Budget guard: PASS; {'phase': 'hpo', 'jobs': 12, 'target': 15, 'remaining_epochs': 180}
01_bounded_grid: target=15, retained=12, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/semantic/tiny_siamese_second/second_oracle_bounded21_e15_25_40_65_bf16_w4_v1/operations/01_bounded_grid_e15_4522bbcf/plan.json
[RUN] 3b8004e53d9e3a47bef8 seed=42 -> cuda:3, target=15
[RUN] 88cb1c806a172cb740ed seed=42 -> cuda:0, target=15
[RUN] b7259174e128d12d2084 seed=42 -> cuda:1, target=15
[RUN] 309a60c1162fd7ea2099 seed=42 -> cuda:2, target=15
[FRESH] seed=42: epoch 0 -> 15 (+15)
[FRESH] seed=42: epoch 0 -> 15 (+15)
[FRESH] seed=42: epoch 0 -

,seed,saved_epoch,target_epoch,additional_epochs,action,lr,batch,scheduler,dropout
0,42,15,25,10,resume,0.001,8,none,0.0
1,123,0,25,25,fresh,0.001,8,none,0.0
2,777,0,25,25,fresh,0.001,8,none,0.0


Retained: 3 Skipped: 0
Remaining epochs summed over jobs: 60
Plan: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/semantic/tiny_siamese_second/second_oracle_bounded21_e15_25_40_65_bf16_w4_v1/operations/seeds_e25_1e3de749/plan.json
Budget guard: PASS; {'phase': 'seeds', 'jobs': 3, 'target': 25, 'remaining_epochs': 60}
01_bounded_grid: target=15, retained=12, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/semantic/tiny_siamese_second/second_oracle_bounded21_e15_25_40_65_bf16_w4_v1/operations/01_bounded_grid_e15_4522bbcf/plan.json
seeds: target=25, retained=3, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/semantic/tiny_siamese_second/second_oracle_bounded21_e15_25_40_65_bf16_w4_v1/operations/seeds_e25_1e3de749/plan.json
[RUN] 7755010c4d99fc29a9b3 seed=42 -> cuda:3, target=25
[RUN] f9da71a

,rank,seeds,mean_macro_F1_two_dates,sample_std_macro_F1_two_dates,parameters
0,1,"[42, 123, 777]",0.515045,0.007769,"{'batch_size': 8, 'class_weights': None, 'lear..."


... (3 outputs left out)

,seed,saved_epoch,target_epoch,additional_epochs,action,lr,batch,scheduler,dropout
0,42,40,65,25,resume,0.001,8,none,0.0
1,123,40,65,25,resume,0.001,8,none,0.0
2,777,40,65,25,resume,0.001,8,none,0.0


Retained: 3 Skipped: 0
Remaining epochs summed over jobs: 75
Plan: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/semantic/tiny_siamese_second/second_oracle_bounded21_e15_25_40_65_bf16_w4_v1/operations/final_e65_88bcc7ed/plan.json
Budget guard: PASS; {'phase': 'final', 'jobs': 3, 'target': 65, 'remaining_epochs': 75}
seed=42: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/semantic/tiny_siamese_second/second_oracle_bounded21_e15_25_40_65_bf16_w4_v1/runs/40ea6ec95f7ce0d99049/tiny_siamese_change_classifier/seed_42/best.pt
seed=123: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/semantic/tiny_siamese_second/second_oracle_bounded21_e15_25_40_65_bf16_w4_v1/runs/81cc47a555ec61f7cddf/tiny_siamese_change_classifier/seed_123/best.pt
seed=777: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hype

{'parameters': {'batch_size': 8,
  'class_weights': None,
  'learning_rate': 0.001,
  'model_parameters': {'classes': ['non-vegetated ground surface',
    'tree',
    'low vegetation',
    'water',
    'buildings',
    'playgrounds'],
   'dropout': 0.0},
  'num_workers': 4,
  'precision': 'bf16',
  'prepared_identity': 'eaa5c557df33a35e2f11039fd36fc1b917e2a1223d7eeaef6eb7582e0c084b79',
  'scheduler': 'none',
  'semantic_config': {'class_ids': {'1': 'non-vegetated ground surface',
    '2': 'tree',
    '3': 'low vegetation',
    '4': 'water',
    '5': 'buildings',
    '6': 'playgrounds'},
   'classes': ['non-vegetated ground surface',
    'tree',
    'low vegetation',
    'water',
    'buildings',
    'playgrounds'],
   'detector': None,
   'ignore_ids': [0],
   'label_policy': {'min_transition_purity': 0.8, 'min_valid_fraction': 0.8},
   'mask_source': 'ground_truth',
   'roi': {'connectivity': 8,
    'context_pixels': 16,
    'min_component_area': 1,
    'min_crop_size': 128,
    'thre

Test не используется для выбора. MODE='benchmark' запускает отдельную оценку.


## 9. Состояние поиска и validation-результаты

Чтение JSON без повторного обучения/оценки. При необходимости эту ячейку можно запускать
отдельно после ошибки. Отсутствующая метрика не подменяется нулём.


In [12]:
def walk_json(value, path=()):
    yield path, value
    if isinstance(value, dict):
        for key, child in value.items():
            yield from walk_json(child, (*path, str(key)))
    elif isinstance(value, list):
        for index, child in enumerate(value):
            yield from walk_json(child, (*path, str(index)))


def show_search_diagnostics(directory):
    files = sorted((Path(directory) / "operations").glob("*/results.json"))
    if not files:
        print("operations/*/results.json пока отсутствуют:", directory)
        return
    status_rows, failures = [], []
    for path in files:
        try:
            payload = read_json(path)
        except (OSError, json.JSONDecodeError) as exc:
            print("Не удалось прочитать", path, repr(exc))
            continue
        # Читаем записи со status, игнорируем вложенную историю попыток.
        records = [(p, value) for p, value in walk_json(payload)
                   if isinstance(value, dict) and isinstance(value.get("status"), str)
                   and not any(x.lower() in {"attempts", "history", "historical_failures"} for x in p)]
        counts = Counter(value["status"] for _, value in records)
        status_rows.append({"operation": path.parent.name, "status_counts": dict(counts), "file": str(path)})
        for record_path, value in records:
            if value["status"].lower() in {"failed", "error"}:
                failures.append({"operation": path.parent.name, "json_path": ".".join(record_path),
                                 "error": value.get("error", value.get("exception", "см. results.json"))})
    display(pd.DataFrame(status_rows))
    if failures:
        print("Ошибки в текущих status-записях (не только счётчик):")
        display(pd.DataFrame(failures))
    else:
        print("В распознанных status-записях failed/error нет; это не доказательство завершения всех этапов.")


show_search_diagnostics(SEARCH_DIRECTORY)


,operation,status_counts,file
0,01_bounded_grid_e15_4522bbcf,{'completed': 12},/home/ustenkoroma0-40gmail-2ecom/projects/sate...
1,final_e40_a1941875,{'completed': 3},/home/ustenkoroma0-40gmail-2ecom/projects/sate...
2,final_e65_88bcc7ed,{'completed': 3},/home/ustenkoroma0-40gmail-2ecom/projects/sate...
3,seeds_e25_1e3de749,{'completed': 3},/home/ustenkoroma0-40gmail-2ecom/projects/sate...


В распознанных status-записях failed/error нет; это не доказательство завершения всех этапов.


In [13]:
if RUNTIME_REPORT_PATH.is_file():
    display(pd.DataFrame(read_json(RUNTIME_REPORT_PATH).get("calls", [])))
if SEED_SUMMARY_PATH.is_file():
    _seed_summary = read_json(SEED_SUMMARY_PATH)
    if _seed_summary.get("complete"):
        print(f"Validation на seeds до эпохи {SEED_TARGET}; одна конфигурация:")
        show_seed_ranking(_seed_summary)

_milestone_rows = []
for _phase in ("intermediate", "final"):
    _path = phase_record_path(_phase)
    if not _path.is_file():
        continue
    _record = read_json(_path)
    if _record.get("fingerprint") != BUDGET_FINGERPRINT:
        raise RuntimeError(f"Отчёт другого протокола: {_path}")
    for _result in _record["summary"].get("results", []):
        _parameters = _result["parameters"]
        _row = {"phase": _phase, "seed": _parameters["seed"],
                "completed_epoch": _result["epoch"],
                "learning_rate": _parameters["learning_rate"],
                "weight_decay": _parameters["weight_decay"],
                "dropout": _parameters["model_parameters"]["dropout"]}
        for _json_path, _value in walk_json(_result):
            if any(_token in {"parameters", "history", "training_history", "attempts"} for _token in _json_path):
                continue
            if _json_path and _json_path[-1] in {"selection_score", "joint_accuracy", "macro_f1", "macro_iou"}:
                if isinstance(_value, (int, float)) and not isinstance(_value, bool):
                    _row[".".join(_json_path)] = _value
        _milestone_rows.append(_row)
if _milestone_rows:
    print("Сохранённые промежуточные и итоговые отчёты, без нового inference:")
    display(pd.DataFrame(_milestone_rows))
    print("completed_epoch — достигнутая эпоха. Метрики и пути сохраняют смысл backend;")
    print("best.pt выбран по validation и может соответствовать более ранней эпохе.")


,phase,jobs,target,remaining_epochs,status,wall_seconds,finished_utc
0,hpo,12,15,180,complete,19605.779071,2026-10-09T14:25:31.434215+00:00
1,seeds,3,25,60,complete,8914.177211,2026-10-09T16:54:11.438265+00:00
2,intermediate,3,40,45,complete,4896.641227,2026-10-09T18:15:53.018179+00:00
3,final,3,65,75,complete,7709.127052,2026-10-09T20:24:28.569628+00:00


Validation на seeds до эпохи 25; одна конфигурация:


,rank,seeds,mean_macro_F1_two_dates,sample_std_macro_F1_two_dates,parameters
0,1,"[42, 123, 777]",0.515045,0.007769,"{'batch_size': 8, 'class_weights': None, 'lear..."


Incomplete families (не ранжируются): 0
Selection scope: all_candidates
Сохранённые промежуточные и итоговые отчёты, без нового inference:


,phase,seed,completed_epoch,learning_rate,weight_decay,dropout,selection_score,joint_accuracy,selected_metrics.joint_accuracy,selected_metrics.selection_score,metrics.selection_score,metrics.joint_accuracy
0,intermediate,123,40,0.001,0.001,0.0,0.522538,0.418191,0.418191,0.522538,0.522538,0.418191
1,intermediate,42,40,0.001,0.001,0.0,0.518849,0.423950,0.423950,0.518849,0.518849,0.423950
2,intermediate,777,40,0.001,0.001,0.0,0.513526,0.416497,0.416497,0.513526,0.513526,0.416497
3,final,123,65,0.001,0.001,0.0,0.522538,0.418191,0.418191,0.522538,0.522538,0.418191
4,final,42,65,0.001,0.001,0.0,0.518849,0.423950,0.423950,0.518849,0.518849,0.423950
5,final,777,65,0.001,0.001,0.0,0.513526,0.416497,0.416497,0.513526,0.513526,0.416497


completed_epoch — достигнутая эпоха. Метрики и пути сохраняют смысл backend;
best.pt выбран по validation и может соответствовать более ранней эпохе.


## 10. Benchmark — после обучения, отдельным запуском

`MODE="benchmark"` → Run All: validation → test → скорость → export через существующий
`run_final_benchmark`. Новых training trials здесь нет, но повторная оценка датасета занимает
время. После достижения FINAL_TARGET все три checkpoint передаются с настоящими seed IDs; deployment seed заранее равен 42.

Результаты — семантическая ROI-классификация, не пространственный IoU/F1 change-mask.
Macro IoU не вычисляется преобразованием усреднённого F1. Все размеры native, первый validation
ROI в speed benchmark не представляет всю совокупность ROI и не является latency каскада.

Сохранены `strict_validation=True` и `overwrite=False`. Совпавшие готовые отчёты читаются без
повторной оценки. Test не используется в HPO и для выбора checkpoint/seed.


In [14]:
ARTIFACT_DIRECTORY = PROJECT_ROOT / "artifacts/final_models" / VARIANT
BENCHMARK_RECORD_PATH = SEARCH_DIRECTORY / "notebook_benchmark_record.json"
_benchmark_result = None
if MODE == "benchmark":
    if not FINAL_SUMMARY_PATH.is_file():
        raise FileNotFoundError("Нет final_training_summary.json. Сначала заверши final на трёх seeds.")
    # Перечитываем с диска: не полагаемся на объекты старого kernel.
    final_summary = read_json(FINAL_SUMMARY_PATH)
    if completed_target(final_summary) != FINAL_TARGET:
        raise RuntimeError(f"Нужны все три seed до {FINAL_TARGET}; промежуточные 40 эпох — не конец эксперимента.")
    checkpoints = checked_final(final_summary, target=FINAL_TARGET)
    benchmark_options = dict(
        task="semantic_roi_classification",
        variant=VARIANT,
        checkpoints=checkpoints,
        project_root=PROJECT_ROOT,
        dataset_root=PREPARED,
        training_results=FINAL_SUMMARY_PATH,
        deployment_seed=DEPLOYMENT_SEED,
        device=f"cuda:{GPU_IDS[0]}",
        evaluation_batch_size=1,
        num_workers=NUM_WORKERS,
        precision=EVALUATION_PRECISION or BASE["precision"],
        warmup=BENCHMARK_WARMUP,
        repetitions=BENCHMARK_REPETITIONS,
        strict_validation=True,
        overwrite=False,
    )
    inspect.signature(run_final_benchmark).bind(**benchmark_options)
    benchmark_identity = {
        "variant": VARIANT, "checkpoints": {str(seed): {"path": str(path.resolve()), "sha256": sha256_file(path)}
                                             for seed, path in checkpoints.items()},
        "manifest_sha256": sha256_file(PREPARED), "deployment_seed": DEPLOYMENT_SEED,
        "final_target": FINAL_TARGET, "classes": list(CLASSES),
        "precision": benchmark_options["precision"], "tf32": BASE.get("tf32"),
        "warmup": BENCHMARK_WARMUP, "repetitions": BENCHMARK_REPETITIONS,
        "torch": str(torch.__version__), "gpu": torch.cuda.get_device_name(GPU_IDS[0]),
        "source_hashes": SOURCE_HASHES,
    }
    benchmark_fingerprint = canonical_hash(benchmark_identity)
    prior = read_json(BENCHMARK_RECORD_PATH) if BENCHMARK_RECORD_PATH.is_file() else None
    required_reports = ("validation_results.json", "final_test_results.json", "benchmark.json", "metadata.json", "deployment_weights.pt")
    can_read_existing = (
        prior is not None and prior.get("fingerprint") == benchmark_fingerprint
        and Path(prior["output_dir"]).resolve() == ARTIFACT_DIRECTORY.resolve()
        and all((ARTIFACT_DIRECTORY / filename).is_file() for filename in required_reports)
        and all(prior.get("report_sha256", {}).get(filename) == sha256_file(ARTIFACT_DIRECTORY / filename)
                for filename in required_reports)
    )
    if can_read_existing:
        print("Тот же benchmark уже завершён: читаем артефакты, test повторно не запускается.")
    else:
        if ARTIFACT_DIRECTORY.exists() and any(ARTIFACT_DIRECTORY.iterdir()):
            raise RuntimeError(
                f"{ARTIFACT_DIRECTORY} уже содержит артефакты без совпадающей notebook-записи. "
                "Они не будут перезаписаны или объявлены совместимыми. "
                "Для нового зафиксированного эксперимента используй отдельный VARIANT."
            )
        _benchmark_result = run_final_benchmark(**benchmark_options)
        ARTIFACT_DIRECTORY = Path(_benchmark_result["output_dir"]).resolve()
        missing_reports = [name for name in required_reports if not (ARTIFACT_DIRECTORY / name).is_file()]
        if missing_reports:
            raise RuntimeError(f"Benchmark вернул output_dir, но отсутствуют ожидаемые артефакты: {missing_reports}. Проверь версию semantic benchmark.")
        write_json_atomic(BENCHMARK_RECORD_PATH, {
            "fingerprint": benchmark_fingerprint, "identity": benchmark_identity,
            "output_dir": str(ARTIFACT_DIRECTORY),
            "report_sha256": {name: sha256_file(ARTIFACT_DIRECTORY / name) for name in required_reports},
        })
    print("Final artifacts:", ARTIFACT_DIRECTORY)
else:
    print("Test evaluation отключена. После final выбери MODE='benchmark'.")


Test evaluation отключена. После final выбери MODE='benchmark'.


## 11. Сохранённые benchmark-отчёты

Исходные названия scalar metrics, mean/sample std по трём seeds, per-class/confusion tables
и speed/memory. Готовые отчёты исходного большого эксперимента не смешиваются с quick baseline.


In [15]:
_METRIC_NAMES = {
    "selection_score", "macro_f1", "macro_iou", "f1_macro", "iou_macro",
    "joint_accuracy", "accuracy", "macro_precision", "macro_recall",
    "mean_macro_f1", "mean_macro_iou", "before_macro_f1", "after_macro_f1",
}


def semantic_metric_leaves(value, path=(), seed=None):
    """Read numeric metric leaves; preserve their original semantic JSON paths."""
    if isinstance(value, dict):
        own_seed = value.get("seed")
        if own_seed is None and isinstance(value.get("parameters"), dict):
            own_seed = value["parameters"].get("seed")
        if own_seed is not None and str(own_seed) in {str(s) for s in FINAL_SEEDS}:
            seed = int(own_seed)
        for key, child in value.items():
            if str(key).lower() in {"parameters", "config", "configuration", "metadata", "history", "training_history"}:
                continue
            child_seed = int(key) if str(key) in {str(s) for s in FINAL_SEEDS} else seed
            # Remove only a confirmed seed-key from the metric name.
            child_path = path if str(key) in {str(s) for s in FINAL_SEEDS} else (*path, str(key))
            yield from semantic_metric_leaves(child, child_path, child_seed)
    elif isinstance(value, list):
        for index, child in enumerate(value):
            # List indices identify records, not metric names, only for explicit seed records.
            is_seed_record = isinstance(child, dict) and (
                "seed" in child or isinstance(child.get("parameters"), dict) and "seed" in child["parameters"]
            )
            yield from semantic_metric_leaves(child, path if is_seed_record else (*path, str(index)), seed)
    elif isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(float(value)):
        if path and path[-1].lower() in _METRIC_NAMES:
            yield {"seed": seed, "metric": ".".join(path), "value": float(value)}


_reports = {}
_report_binding = read_json(BENCHMARK_RECORD_PATH) if BENCHMARK_RECORD_PATH.is_file() else None
_reports_match = False
if _report_binding and FINAL_CHECKPOINTS and PREPARED:
    identity = _report_binding.get("identity", {})
    current_checkpoint_ids = {str(seed): {"path": str(path.resolve()), "sha256": sha256_file(path)}
                              for seed, path in FINAL_CHECKPOINTS.items()}
    _reports_match = (
        Path(_report_binding["output_dir"]).resolve() == ARTIFACT_DIRECTORY.resolve()
        and identity.get("variant") == VARIANT
        and identity.get("checkpoints") == current_checkpoint_ids
        and identity.get("manifest_sha256") == sha256_file(PREPARED)
        and identity.get("classes") == list(CLASSES)
        and identity.get("final_target") == FINAL_TARGET
        and identity.get("deployment_seed") == DEPLOYMENT_SEED
        and identity.get("source_hashes") == SOURCE_HASHES
    )
_report_names = ("validation_results.json", "final_test_results.json", "benchmark.json", "metadata.json")
if _reports_match:
    for filename in _report_names:
        path = ARTIFACT_DIRECTORY / filename
        if not path.is_file() or _report_binding.get("report_sha256", {}).get(filename) != sha256_file(path):
            raise RuntimeError(f"Сохранённый отчёт отсутствует или изменился: {path}. Он не будет переоценён/перезаписан автоматически.")
        _reports[filename] = read_json(path)

if not _reports:
    print("Нет финальных отчётов, подтверждённо связанных с текущими checkpoints/target:", ARTIFACT_DIRECTORY)
    if ARTIFACT_DIRECTORY.is_dir() and any(ARTIFACT_DIRECTORY.iterdir()):
        print("Существующие файлы оставлены без изменений; их метрики не выданы за текущий эксперимент.")
else:
    print("Доступные файлы:")
    display(pd.DataFrame([{"file": p.name, "KiB": round(p.stat().st_size / 1024, 2)}
                          for p in sorted(ARTIFACT_DIRECTORY.iterdir()) if p.is_file()]))
    all_metrics = []
    for filename in ("validation_results.json", "final_test_results.json"):
        if filename not in _reports:
            continue
        rows = list(semantic_metric_leaves(_reports[filename]))
        for row in rows:
            row["split_report"] = filename
        all_metrics.extend(rows)
        print("\n", filename)
        if rows:
            display(pd.DataFrame(rows))
        else:
            print("Scalar schema не распознана; исходный JSON показан без подмены метрик:")
            display(_reports[filename])
    if all_metrics:
        metrics_long = pd.DataFrame(all_metrics)
        mean_std_rows = []
        for (split_report, metric), group in metrics_long.dropna(subset=["seed"]).groupby(["split_report", "metric"]):
            if len(group) == len(FINAL_SEEDS) and set(group["seed"].astype(int)) == set(FINAL_SEEDS):
                mean_std_rows.append({
                    "split_report": split_report, "metric": metric, "n_seeds": len(group),
                    "mean": float(group["value"].mean()), "sample_std": float(group["value"].std(ddof=1)),
                })
        print("\nНезависимая сводка по трём seed-значениям (только распознанные поля):")
        if mean_std_rows:
            display(pd.DataFrame(mean_std_rows))
        else:
            print("Нет полного набора per-seed scalar полей; готовые агрегаты смотри в исходном JSON.")
        metrics_long.to_csv(ARTIFACT_DIRECTORY / "notebook_semantic_metrics_long.csv", index=False)
        if mean_std_rows:
            pd.DataFrame(mean_std_rows).to_csv(ARTIFACT_DIRECTORY / "notebook_semantic_mean_std.csv", index=False)
    if "benchmark.json" in _reports:
        print("\nSpeed / memory — исходные единицы и ROI shape:")
        display(_reports["benchmark.json"])
    if "metadata.json" in _reports:
        print("\nDeployment metadata / provenance:")
        display(_reports["metadata.json"])
    # Матрицы и per-class таблицы не суммируются между seeds в одну «модель».
    for filename in ("validation_results.json", "final_test_results.json"):
        for json_path, value in walk_json(_reports.get(filename, {})):
            if json_path and any(token in json_path[-1].lower() for token in ("confusion", "per_class")) and isinstance(value, (dict, list)):
                print(filename, ".".join(json_path))
                display(value)


Нет финальных отчётов, подтверждённо связанных с текущими checkpoints/target: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/final_models/tiny_siamese_second_oracle_bounded21_e65_bf16_w4_v1


## Пауза, resume и границы результата

Для мягкой паузы вызови из **другого kernel/терминала**:

```python
from pathlib import Path
from satchange.automated_hpo import request_hpo_pause
request_hpo_pause(
    Path("/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence")
    / "artifacts/hyperparameter_search/semantic/tiny_siamese_second"
    / "second_oracle_bounded21_e15_25_40_65_bf16_w4_v1",
    after="epoch",
)
```

При выборе цели 60/70 или другой точности используй фактическую `SEARCH_DIRECTORY`,
напечатанную в начале notebook. Не отправляй pause в старую папку, когда работает новый поиск.
Для прежнего длительного эксперимента путь оканчивается на `second_oracle_hpo_v3_bf16_w4`.
После запроса дождись сохранения checkpoints и завершения workers; новый controller сам
не останавливает процессы другого search.

Для продолжения не меняй настройки: `MODE="train_all"` → Restart Kernel → Run All.
Файловый lock защищает только этот notebook/search. Он не освобождает GPU от чужих задач
и не разрешает запускать два разных поиска на тех же устройствах без учёта ресурсов.

После завершения — `MODE="benchmark"` → Run All. При совпадающих checkpoints и отчётах
benchmark повторно не оценивает test. Изменения гиперпараметров после просмотра test
являются новым исследовательским этапом, а не независимой финальной проверкой.

**Бюджет с нуля:** при final=60 — 345, при final=65 — 360, при final=70 — 375 суммарных
эпох. Во всех трёх случаях 21 плановое задание и 12 уникальных конфигураций. Это лимит
плановых эпох; повтор незаписанной работы после аварии и стоимость validation в него
как отдельные эпохи не входят. Число часов зависит от количества/размера ROI и скорости
всего pipeline, а не только от числа параметров CNN.

### Что сохранено из текущего notebook

Существующие semantic API, полные train/validation SECOND ROI, native pixel scale,
фиксированный split seed, class order из YAML, выбор по validation, три training seeds,
resume из собственных состояний, strict validation benchmark и запрет overwrite.

Изменены организация и бюджет поиска. Не запускаются четыре разрастающихся HPO-блока,
не запускается отдельный 1→2→3 smoke на полном SECOND, нет переустановки датасета или
импорта старых 40-эпоховых результатов в 15-эпоховое сравнение.

### Основа и пределы проверки

Основа: текущий notebook пользователя из `Вставленный текст.txt`, ранее предоставленный
`Tiny_Siamese_SECOND_Quick_HPO_18.ipynb` и `tiny_siamese_change_classifier.py`.
В текущем notebook раздел «Пауза, resume и увеличение бюджета» документирует
`additional_epochs` / `extension_id`; именно этот API используется для шага 40→65.

Синтаксис и управление фазами проверены локально на тестовой замене runner; CNN проверяется
синтетически. Это **не** запуск реального серверного `satchange` на SECOND/A100 и не
подтверждение корректности его внутренней реализации optimizer/scheduler/RNG resume.
Проверки планов останавливают неподдерживаемое поведение, а не обходят защиту backend.

65 эпох дают более длинную кривую обучения, но не гарантируют высокое качество. Оценка
здесь относится к классификации GT/oracle ROI; каскад «детектор → mask → crop → что изменилось»
остаётся следующим отдельным экспериментом. Локальный holdout не называется официальным test.
